## Imports

In [ ]:
# Import pandas, numpy, and matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# seaborn is a data visualization library built on matplotlib
import seaborn as sns
# set the plotting style
sns.set_style("whitegrid")

# Train-test splits
from sklearn.model_selection import train_test_split, GridSearchCV

# Model preprocessing
from sklearn import preprocessing

# Data
from sklearn import datasets

# Models
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.ensemble import RandomForestClassifier
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Model metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, ConfusionMatrixDisplay

## Load the data

##### $\rightarrow$ Load the `EdGap` data set.

In [ ]:
df = pd.read_excel('https://raw.githubusercontent.com/brian-fischer/DATA-5100/main/EdGap_data.xlsx', dtype={'NCESSCH School ID':object})

##### $\rightarrow$ Explore the data set.

In [ ]:
df.head()

In [ ]:
df.info()

##### $\rightarrow$ Rename the columns.

In [ ]:
df = df.rename(columns={"NCESSCH School ID":"id",
              "CT Pct Adults with College Degree":"percent_college",
              "CT Unemployment Rate":"rate_unemployment",
              "CT Pct Childre In Married Couple Family":"percent_married",
              "CT Median Household Income":"median_income",
              "School ACT average (or equivalent if SAT score)":"average_act",
              "School Pct Free and Reduced Lunch":"percent_lunch"})

In [ ]:
df.head()

##### $\rightarrow$ Make a pair plot to explore relationships between the variables.

In [ ]:
#sns.pairplot(df.drop(columns = 'id'));

## Quick data cleaning

##### $\rightarrow$ Check the range of each variable.

In [ ]:
df.agg(['min','max'])

##### $\rightarrow$ Set out-of-range values to `NaN`.

In [ ]:
df.loc[df['average_act'] < 1, 'average_act'] = np.nan

In [ ]:
df.loc[df['percent_lunch'] < 0, 'percent_lunch'] = np.nan

##### $\rightarrow$ Find and replace `NaN` values of **input variables** with the mode of each column.

In [ ]:
df.isna().sum()

In [ ]:
input_modes = df[df.columns.difference(['id','average_act'])].mode().iloc[0]

In [ ]:
df = df.fillna(input_modes)

In [ ]:
df.isna().sum()

##### $\rightarrow$ Drop columns where the output variable `average_act` is `NaN`.

In [ ]:
df = df.dropna()

In [ ]:
df.isna().sum()

## Train test split

##### $\rightarrow$ Define the matrix of predictor variables `X` to be all columns except `id` and `average_act` and define the output variable `y` to be `average_act`.

In [ ]:
X = df[df.columns.difference(['id','average_act'])]
y = df['average_act']

##### $\rightarrow$ Split the data into training and testing sets. Keep 20% of the data for the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2)

##### $\rightarrow$ Scale the predictor variables in the training set to have mean 0 and standard deviation 1.

Define the scaler using only the training data. For the validation set approach to work, we can not incorporate any knowledge of the validation set's properties into the model building process.

In [ ]:
scaler = preprocessing.StandardScaler().fit(X_train)

In [ ]:
print(scaler.mean_, scaler.scale_)

##### $\rightarrow$ Perform the scaling transform on the predictors in the training and testing sets.

In [ ]:
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

## Model comparison

### Linear regression code example

In [ ]:
# Define the model
model_lr = LinearRegression()

# Fit the model
model_lr.fit(X_train, y_train)

# Compute test mean squared error
mse_lr = mean_squared_error(y_test, model_lr.predict(X_test))

# Plot predictions
figure = plt.figure(figsize=(8, 6))

plt.plot(y_test, model_lr.predict(X_test), 'o', alpha = 0.4)
plt.xlabel('Average ACT score',fontsize = 20)
plt.title('Lin. Reg.' + ' ' + ' Test MSE = ' + str(mse_lr.round(3)),fontsize = 14)
plt.ylabel('Predicted Average ACT score',fontsize = 20);

### Neural network code example

In [ ]:
# Define the model
model_net = MLPRegressor()

# Fit the model
model_net.fit(X_train, y_train)

# Compute test mean squared error
mse_net = mean_squared_error(y_test, model_net.predict(X_test))

# Plot predictions
figure = plt.figure(figsize=(8, 6))
#plt.subplot(1,2,1)
plt.plot(y_test, model_net.predict(X_test), 'o', alpha = 0.4)
plt.xlabel('Average ACT score',fontsize = 20)
plt.title('Neural net' + ' ' + ' Test MSE = ' + str(mse_net.round(3)),fontsize = 14)
plt.ylabel('Predicted Average ACT score',fontsize = 20);

In [ ]:
plt.figure(figsize = (18,4))
plt.imshow(model_net.coefs_[0])
plt.colorbar();

Create a neural network with an additional hidden layer of 50 neurons.

In [ ]:
# Define the model
model = MLPRegressor(hidden_layer_sizes = (100, 50))

# Fit the model
model.fit(X_train, y_train)

# Compute test mean squared error
mse = mean_squared_error(y_test, model.predict(X_test))

# Plot predictions
figure = plt.figure(figsize=(8, 6))
#plt.subplot(1,2,1)
plt.plot(y_test, model.predict(X_test), 'o', alpha = 0.4)
plt.xlabel('Average ACT score',fontsize = 20)
plt.title('Neural net' + ' ' + ' Test MSE = ' + str(mse.round(3)),fontsize = 14)
plt.ylabel('Predicted Average ACT score',fontsize = 20);

In [ ]:
plt.subplots(3,1, figsize = (18,4))

plt.subplot(3,1,1)
plt.imshow(model.coefs_[0])
plt.colorbar();

plt.subplot(3,1,2)
plt.imshow(model.coefs_[1])
plt.colorbar();

plt.subplot(3,1,3)
plt.plot(model.coefs_[2],'o')
plt.colorbar();

### Cross validation

In [ ]:
# Define base regressor:
base_reg = MLPRegressor()

# Define search space:
params = {
    'hidden_layer_sizes': [
        (50,), (100,),
        (100, 50), (20, 10), (30, 10),
        (40, 10), (90, 10), (90, 30, 10)
    ]
}

# Find best hyper params and then refit on all training data:
reg = GridSearchCV(estimator=base_reg, param_grid=params, n_jobs=8, cv=3, refit=True, verbose=5)

reg.fit(X_train, y_train)

print(reg.best_estimator_)

In [ ]:
# Define the model
model = MLPRegressor(hidden_layer_sizes=(50,))

# Fit the model
model.fit(X_train, y_train)

# Compute test mean squared error
mse = mean_squared_error(y_test, model.predict(X_test))

# Plot predictions
figure = plt.figure(figsize=(8, 6))
#plt.subplot(1,2,1)
plt.plot(y_test, model.predict(X_test), 'o', alpha = 0.4)
plt.xlabel('Average ACT score',fontsize = 20)
plt.title('Neural net' + ' ' + ' Test MSE = ' + str(mse.round(3)),fontsize = 14)
plt.ylabel('Predicted Average ACT score',fontsize = 20);